**Code:** [GitHub · Kukai-7/industrial-data-valuation](https://github.com/Kukai-7/industrial-data-valuation) · **Synthetic data:** [Hugging Face · synthetic-industrial-data-valuation](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · **Demo:** [Space](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)

**Abstract.** How can the value of an industrial data portfolio, treated as an intangible asset, be ranked? Eight anonymised datasets are scored on 17 compliance criteria, then valued with three approaches (cost, market, income) blended by confidence, with sensitivity analysis, Monte Carlo simulation and AHP weighting. Limits first: amounts are working assumptions giving a relative ranking, not an opposable price.

*Résumé (FR). Classer la valeur d'un portefeuille de données industrielles traité comme actif incorporel : 17 critères de conformité, trois approches, sensibilité, Monte Carlo et AHP, sur un cas anonymisé. Classement relatif, pas un prix opposable.*

<div style="padding:28px 0 6px 0">
<div style="font-size:12px;letter-spacing:.14em;text-transform:uppercase;color:#898781">Research document · October 2026</div>
<h1 style="font-size:34px;line-height:1.1;margin:10px 0 6px 0">Valuing a portfolio of industrial data</h1>
<div style="font-size:17px;color:#6E6E73">Compliance grid, hybrid valuation, regulatory exposure, uncertainty and academic grounding</div>
</div>

This notebook holds **the whole study** in one executable file: assumptions, model, results, limits and sources. Every chart is recomputed from the parameter cells, so any change flows through from start to finish.

**How to read it.** Grey cells hold the code, the others the reasoning. To recompute everything: *Run All* (or `Ctrl + Shift + P`, then "Run All Cells" in VS Code). Packages: `numpy`, `pandas`, `matplotlib` (see `requirements.txt`).

| | Part | What it establishes |
|---|---|---|
| 1 | Data and engine | The 17 criteria, the 8 datasets, the parameters, the formulas |
| 2 | Summary | The four numbers that sum up the study |
| 3 | Framework | The three-step path and where the grid fits |
| 4 | Compliance | Who clears the thresholds, and why |
| 5 | Valuation | Three approaches blended by confidence |
| 6 | Correction | Compliance sorts value without inflating it |
| 7 | Attribution | What actually accrues to the group |
| 8 | Exposure and priorities | Where to act first |
| 9 | Gains by criterion | Which compliance point pays most |
| 10 | Sensitivity | The assumptions that move the result |
| 11 | Monte Carlo | A range rather than a point |
| 12 | AHP weighting | Deriving the weights instead of setting them |
| 13 | External anchors | Market comparables, perceived versus recognised value, market context |
| 14 | Future flows | The value of data from an installed base |
| 15 | Regulatory watch | The timeline up to October 2026 |
| 16 | Literature | Convergence with the state of the art |
| 17 | Limits | What the study does not prove |

> **Reading rule.** All values are **relative**: they rank the datasets against each other, they do not set a price. A value is not a price: a price needs an external benchmark.

## Setup

One palette for the whole document: a blue for magnitude, three fixed hues for the three valuation approaches, and a status palette (green, amber, red) kept for compliance verdicts, always paired with a label.

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ── Palette (colour-blind checked, light mode) ─────────────────────────────
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE, CARD = "#e1e0d9", "#c3c2b7", "#fcfcfb", "#f3f2ee"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"            # cost, market, income
APPROACHES = {"Cost": BLUE, "Market": ORANGE, "Income": AQUA}
GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"                # status
RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
UP, DOWN = "#256abf", "#d03b3b"                                    # diverging
PRIO = {"P1": "#0d366b", "P2": "#256abf", "P3": "#5598e7", "P4": "#86b6ef"}

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 10.5, "axes.titlesize": 14, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.titlepad": 26, "axes.labelcolor": INK2,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "legend.frameon": False, "axes.axisbelow": True,
})

def title(ax, t, sub=None):
    # Left-aligned title + grey subtitle (one sentence saying what to read).
    ax.set_title(t, loc="left", color=INK)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, color=INK2, fontsize=10, va="bottom")

def fig_title(fig, ax, t, sub=None, top=.84):
    # Title placed at figure level (for heatmaps with the x axis on top).
    fig.subplots_adjust(top=top)
    x0 = ax.get_position().x0
    fig.text(x0, .985, t, fontsize=14, fontweight="semibold", color=INK, va="top")
    if sub: fig.text(x0, .945, sub, fontsize=10, color=INK2, va="top")

def eur(v):
    a = abs(v); s = "−" if v < 0 else ""
    if a >= 1e6: return f"{s}€{a/1e6:,.2f}M"
    if a >= 1e3: return f"{s}€{a/1e3:,.0f}k"
    return f"{s}€{a:,.0f}"

def pct(v, d=1):
    return f"{v*100:+.{d}f}%".replace("-", "−")

def table(df, fmt=None, bar=None):
    # Table readable in light and dark themes.
    sty = df.style.hide(axis="index")
    if fmt: sty = sty.format(fmt)
    sty = sty.set_table_styles([
        {"selector": "th", "props": "text-align:left;font-weight:600;padding:6px 12px;border-bottom:1px solid #c3c2b7"},
        {"selector": "td", "props": "padding:5px 12px;border-bottom:1px solid #e1e0d9"},
        {"selector": "", "props": "border-collapse:collapse;font-size:12.5px"},
    ])
    if bar: sty = sty.bar(subset=bar, color="#9ec5f4", vmin=0)
    return sty

print("Ready.")

## 1 · Input data and engine

### 1.1 The 17-criterion grid

The criteria follow the structure of the national guidelines on building high-quality datasets (August 2025): **documentation** (4), **intrinsic quality** (8), **fitness for the model** (5). Each criterion is scored from 1 to 5. Only the last criterion, fitness for the model, is proven by a test rather than a statement, hence the high weight of block C.

In [ ]:
CRITERIA = pd.DataFrame([
    ("A1","Basic information","A","Volume, format, structure, access channel"),
    ("A2","Content characteristics","A","Modalities, distribution, classes, limits"),
    ("A3","Construction process","A","Sources, collection, processing, versions"),
    ("A4","Usage notice","A","Licence, use cases, reference results"),
    ("B1","Format compliance","B","Standard format, directly usable"),
    ("B2","Security compliance","B","No unlawful content, no infringement of rights"),
    ("B3","Annotation compliance","B","Written rule, applied end to end"),
    ("B4","Structural completeness","B","Complete metadata, missing values under control"),
    ("B5","Content accuracy","B","Traceable to the source"),
    ("B6","Content consistency","B","Agreement between linked data"),
    ("B7","Type consistency","B","Matches the level of specialisation claimed"),
    ("B8","Content cleanliness","B","No duplicates, no noise"),
    ("C1","Content diversity","C","Covers the use case, no blind spot"),
    ("C2","Scale completeness","C","Enough volume for the target model"),
    ("C3","Freshness","C","Collection and updates fit for use"),
    ("C4","Annotation accuracy","C","Annotation captures all useful information"),
    ("C5","Fitness for the model","C","Performance gain measured by a test"),
], columns=["code","criterion","block","what it checks"])
BLOCKS = {"A": "Documentation", "B": "Intrinsic quality", "C": "Fitness for the model"}
table(CRITERIA)

### 1.2 The eight datasets

The anonymised portfolio of a **large industrial site**, split between a wholly owned subsidiary and a 50% joint venture. Amounts are working assumptions, consistent with each other, not accounting data.

In [ ]:
DATASETS = pd.DataFrame([
 # dataset, short, entity, share, FTE-months, FTE cost, infra, volume, ref. price, discount, market conf., profit/yr, savings/yr, horizon, v1 factor
 ("Production sensor telemetry","Telemetry","Subsidiary 100%",1.0,18,9000,120000,500000,0.8,0.40,0.55,150000,400000,5,1.00),
 ("Maintenance history","Maintenance","Joint venture",0.5,30,9000,200000,1200000,1.1,0.45,0.50,300000,600000,5,1.10),
 ("Supplier quality and defects","Suppliers","Subsidiary 100%",1.0,10,8500,60000,300000,0.6,0.50,0.45,80000,250000,5,0.90),
 ("Plant energy logs","Energy","Subsidiary 100%",1.0,8,8000,40000,800000,0.2,0.55,0.50,40000,180000,5,0.85),
 ("Supply chain","Supply","Joint venture",0.5,14,8800,90000,600000,0.9,0.45,0.55,200000,350000,5,1.00),
 ("Process parameters","Process","Joint venture",0.5,25,9500,110000,150000,2.5,0.50,0.40,250000,700000,7,1.15),
 ("Personnel data","Personnel","Subsidiary 100%",1.0,6,8000,30000,50000,1.5,0.60,0.35,20000,90000,5,0.80),
 ("Testing and research","Testing","Joint venture",0.5,40,10000,250000,200000,3.0,0.55,0.40,400000,900000,7,1.20),
], columns=["dataset","short","entity","share","fte","fte_cost","infra","vol","price","disc","mkt_conf","profit","savings","horizon","fq1"])

# Scores on the 17 criteria (rows = criteria A1…C5, columns = datasets)
SCORES = pd.DataFrame({
 "Telemetry":   [2,2,1,1,4,4,1,3,4,3,3,2,3,5,5,1,2],
 "Maintenance": [4,4,3,4,4,4,4,4,5,4,5,4,4,4,4,4,4],
 "Suppliers":   [2,2,2,2,3,3,2,2,3,2,3,3,3,3,3,2,3],
 "Energy":      [2,2,1,1,4,5,1,3,4,3,1,3,1,4,5,1,1],
 "Supply":      [3,3,2,3,3,3,3,3,3,3,3,3,3,4,4,3,3],
 "Process":     [4,4,4,3,5,4,4,4,5,4,5,4,3,2,4,4,4],
 "Personnel":   [2,2,2,1,3,1,1,2,3,2,2,2,1,1,3,1,1],
 "Testing":     [4,4,4,4,4,3,4,4,5,4,5,4,5,3,4,4,5],
}, index=CRITERIA.code)

# Regulatory exposure: 5 criteria scored from 1 to 5
EXPO_WEIGHTS = {"Local industry maturity": .30, "Operational criticality": .20,
                "Personal data": .20, "National security": .20, "Joint-venture sharing": .10}
EXPO = pd.DataFrame([[3,4,1,2,2],[5,4,2,3,4],[2,3,1,1,2],[3,2,1,3,1],
                     [3,3,2,2,4],[4,5,1,4,5],[2,2,5,2,1],[4,5,2,5,5]],
                    index=DATASETS.short, columns=list(EXPO_WEIGHTS))

view = DATASETS[["dataset","entity","share","fte","infra","vol","price","profit","savings","horizon"]].copy()
view.columns = ["Dataset","Ownership","Share","FTE-months","Infra (€)","Volume","Ref. price (€)","Profit/yr (€)","Savings/yr (€)","Horizon (yrs)"]
table(view, {"Share":"{:.0%}","Infra (€)":"{:,.0f}","Volume":"{:,.0f}","Profit/yr (€)":"{:,.0f}","Savings/yr (€)":"{:,.0f}","Ref. price (€)":"{:.1f}"})

### 1.3 Model parameters

All adjustable here. The three approach weights (30 / 25 / 45) are set by judgement; part 12 shows how to derive them properly.

In [ ]:
P = dict(
    wDoc=.20, wQual=.45, wApp=.35,          # weights of the three blocks of the grid
    fMin=.70, fMax=1.30,                     # bounds of the quality factor (cost approach)
    thrCompliant=75, thrReserved=55,         # verdicts
    alpha=.30, beta=.50,                     # effect of the score on the discount and on income
    discMin=.05, discMax=.80,
    rate=.12,                                # discount rate for income
    pc=.30, pm=.25, pr=.45,                  # weights of the cost / market / income approaches
    cc=.90, cr=.70,                          # confidence in cost / income (market: per dataset)
    thrValue=50, thrExpo=50,                 # prioritisation
)
pd.Series(P).to_frame("value").T

### 1.4 The calculation engine

The whole model fits in the functions below. The compliance score acts on **all three** approaches at once: it scales the quality factor of the cost approach, lowers the market discount and raises the income multiplier.

$$V_{\text{hybrid}} = \frac{p_c\,c_c\,V_c \;+\; p_m\,c_m\,V_m \;+\; p_r\,c_r\,V_r}{p_c\,c_c + p_m\,c_m + p_r\,c_r}$$

In [ ]:
def clip(x, a, b): return np.minimum(b, np.maximum(a, x))

def block_score(scores, block):
    idx = CRITERIA.index[CRITERIA.block == block]
    return (scores.iloc[idx].mean() - 1) / 4 * 100

def global_score(scores, P=P):
    return P["wDoc"]*block_score(scores,"A") + P["wQual"]*block_score(scores,"B") + P["wApp"]*block_score(scores,"C")

def annuity(h, t): return (1 - (1 + t) ** (-h)) / t

def value(j, s, P=P, mode="after"):
    # Returns the three approaches and the hybrid value of dataset j for score s.
    if mode == "v1":
        f, disc, mul = j.fq1, j.disc, 1.0
    else:
        f    = clip(P["fMin"] + (s - 20) / 80 * (P["fMax"] - P["fMin"]), P["fMin"], P["fMax"])
        disc = clip(j.disc - P["alpha"] * (s - 50) / 100, P["discMin"], P["discMax"])
        mul  = 1 + P["beta"] * (s - 50) / 100
    vc = (j.fte * j.fte_cost + j.infra) * f
    vm = j.vol * j.price * (1 - disc)
    vr = (j.profit + j.savings) * mul * annuity(j.horizon, P["rate"])
    wc, wm, wr = P["pc"]*P["cc"], P["pm"]*j.mkt_conf, P["pr"]*P["cr"]
    den = wc + wm + wr
    return dict(vc=vc, vm=vm, vr=vr, val=(wc*vc + wm*vm + wr*vr)/den,
                part_c=wc*vc, part_m=wm*vm, part_r=wr*vr, den=den, f=f, disc=disc, mul=mul)

def verdict(s, P=P):
    return "Compliant" if s >= P["thrCompliant"] else ("With reservations" if s >= P["thrReserved"] else "Non-compliant")

def compute(P=P, D=DATASETS, S=SCORES):
    rows = []
    for _, j in D.iterrows():
        s = global_score(S[j.short], P)
        bf, af = value(j, s, P, "v1"), value(j, s, P, "after")
        rows.append(dict(short=j.short, dataset=j.dataset, entity=j.entity, share=j.share, score=s,
                         A=block_score(S[j.short],"A"), B=block_score(S[j.short],"B"), C=block_score(S[j.short],"C"),
                         verdict=verdict(s, P), before=bf["val"], after=af["val"],
                         vc=af["vc"], vm=af["vm"], vr=af["vr"],
                         part_c=af["part_c"]/(af["val"]*af["den"]), part_m=af["part_m"]/(af["val"]*af["den"]), part_r=af["part_r"]/(af["val"]*af["den"]),
                         factor=af["f"], attrib=af["val"]*j.share, attrib_before=bf["val"]*j.share))
    R = pd.DataFrame(rows).set_index("short")
    R["gap"] = R.after / R.before - 1
    return R

R = compute()
VERDICT_COL = {"Compliant": GOOD, "With reservations": WARN, "Non-compliant": CRIT}
TOT_BF, TOT_AF = R.attrib_before.sum(), R.attrib.sum()
print(f"Attributable portfolio: {eur(TOT_BF)} → {eur(TOT_AF)}  ({pct(TOT_AF/TOT_BF-1)})")

## 2 · Summary

The four numbers to remember, recomputed at every run.

In [ ]:
fig = plt.figure(figsize=(12, 2.9))
tiles = [
    (f"{len(R)}", "datasets assessed", "on 17 criteria each", INK),
    (f"{(R.verdict=='Compliant').sum()} / {len(R)}", "compliant datasets", "the only ones fit to present without reservation", GOOD),
    (eur(TOT_AF), "attributable value", f"portfolio {pct(TOT_AF/TOT_BF-1)} after correction", BLUE),
    (f"{pct(R.gap.min(),0)} to {pct(R.gap.max(),0)}", "correction per dataset", "the method sorts, it does not inflate", INK),
]
for i, (v, l, s, c) in enumerate(tiles):
    ax = fig.add_axes([0.005 + i*0.25, 0.04, 0.235, 0.9]); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.06",
                                transform=ax.transAxes, fc=CARD, ec="none"))
    ax.text(.08, .66, v, fontsize=25, fontweight="semibold", color=c, transform=ax.transAxes)
    ax.text(.08, .36, l, fontsize=11.5, color=INK, transform=ax.transAxes)
    ax.text(.08, .17, s, fontsize=9.5, color=MUTED, transform=ax.transAxes)
plt.show()

**In one sentence:** of eight datasets, only two clear the compliance threshold; the quality correction moves each dataset by −16% to +12% but the portfolio by only +4.5%. The method does not create value, it moves it towards the datasets that deserve it.

## 3 · The framework: a three-step path

The director of the national data development research academy describes a three-stage path to value: **resourcing** (inventory, governance), **productisation** (deliverable datasets, APIs, model services) and **assetisation** (rights, balance sheet, circulation). The grid covers the first and the third step. The second remains to be built: it is the natural next step of this work.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 4)
steps = [("1 · Resourcing", "Inventory, govern,\nmake traceable", "Report on 17 criteria", BLUE, False),
         ("2 · Productisation", "Deliverable datasets, APIs,\nmodel services", "To be built", MUTED, True),
         ("3 · Assetisation", "Rights, balance sheet,\ncirculation", "Valuation grid", BLUE, False)]
for i, (t, d, tool, c, empty) in enumerate(steps):
    x = 0.2 + i * 4
    ax.add_patch(FancyBboxPatch((x, 1.9), 3.5, 1.8, boxstyle="round,pad=0.02,rounding_size=0.18",
                                fc=SURFACE if empty else CARD, ec=c, lw=1.4, ls="--" if empty else "-"))
    ax.text(x + .25, 3.25, t, fontsize=13, fontweight="semibold", color=INK)
    ax.text(x + .25, 2.15, d, fontsize=10, color=INK2, va="bottom")
    ax.add_patch(FancyBboxPatch((x, .3), 3.5, 1.1, boxstyle="round,pad=0.02,rounding_size=0.18",
                                fc=SURFACE, ec=c, lw=1.0, ls=":" if empty else "-"))
    ax.text(x + 1.75, .85, tool, fontsize=10.5, ha="center", va="center",
            color=WARN if empty else INK, fontweight="semibold" if empty else "normal")
    ax.annotate("", (x + 1.75, 1.42), (x + 1.75, 1.88), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1))
    if i < 2:
        ax.annotate("", (x + 3.95, 2.8), (x + 3.55, 2.8), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1.2))
ax.text(0.2, 3.95, "National path to data value", fontsize=14, fontweight="semibold")
plt.show()

> **An essential nuance.** A high score shows that the data is *ready* to become an asset, not that it has become one. Actual assetisation requires two acts the grid does not measure: registration of property rights and recognition on the balance sheet. It is the phase a national data exchange calls "the most critical" in the first closed-loop case observed (August 2026).

## 4 · Compliance: who clears the thresholds

### 4.1 The score map

Each cell is a score from 1 to 5. Columns are sorted by global score. The contrast is visible at a glance: the best-scored datasets score well everywhere, the weakest fall behind mostly on documentation and annotation.

In [ ]:
order = R.score.sort_values(ascending=False).index
M = SCORES[order]
cmap = mpl.colors.ListedColormap(["#e9f1fb", "#b7d3f6", "#6da7ec", "#2a78d6", "#104281"])
fig, ax = plt.subplots(figsize=(11.5, 7.2))
ax.imshow(M.values, cmap=cmap, vmin=.5, vmax=5.5, aspect="auto")
for (i, k), v in np.ndenumerate(M.values):
    ax.text(k, i, v, ha="center", va="center", fontsize=10, color="white" if v >= 4 else INK)
ax.set_xticks(range(len(order)), [f"{c}\n{R.score[c]:.0f}/100" for c in order], color=INK, fontsize=10)
ax.set_yticks(range(17), [f"{c}  {n}" for c, n in zip(CRITERIA.code, CRITERIA["criterion"])], color=INK2, fontsize=9.5)
ax.xaxis.tick_top(); ax.grid(False)
for y in (3.5, 11.5): ax.axhline(y, color=SURFACE, lw=4)
for sp in ax.spines.values(): sp.set_visible(False)
ax.tick_params(length=0)
fig_title(fig, ax, "The 17 scores of each dataset, from best to worst",
          "Score from 1 (absent) to 5 (compliant and verified) · bands separate documentation, quality, fitness")
plt.show()

### 4.2 The global score against the two thresholds

The score is the weighted mean of the three blocks (20% / 45% / 35%), rescaled to 100. "Compliant" threshold at 75, "with reservations" threshold at 55.

In [ ]:
r = R.sort_values("score")
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.axvspan(0, P["thrReserved"], color=CRIT, alpha=.05, lw=0)
ax.axvspan(P["thrReserved"], P["thrCompliant"], color=WARN, alpha=.08, lw=0)
ax.axvspan(P["thrCompliant"], 100, color=GOOD, alpha=.06, lw=0)
for i, (c, row) in enumerate(r.iterrows()):
    col = VERDICT_COL[row.verdict]
    ax.plot([0, row.score], [i, i], color=GRID, lw=1.2, zorder=1)
    for b, m in (("A", "^"), ("B", "s"), ("C", "D")):
        ax.scatter(row[b], i, marker=m, s=26, color="white", edgecolor=MUTED, lw=.9, zorder=2)
    ax.scatter(row.score, i, s=110, color=col, edgecolor="white", lw=2, zorder=3)
    ax.text(102, i, f"{row.score:.0f}", va="center", fontsize=10.5, fontweight="semibold", color=INK, clip_on=False)
    ax.text(108, i, row.verdict, va="center", fontsize=9.5, color=col if row.verdict != "With reservations" else "#9a6a00", clip_on=False)
ax.set_yticks(range(len(r)), r.index, color=INK)
ax.set_xlim(0, 100); ax.set_ylim(-1.1, len(r) - .5); ax.grid(axis="y", visible=False)
for s, l in ((P["thrReserved"], "threshold 55"), (P["thrCompliant"], "threshold 75")):
    ax.axvline(s, color=AXIS, ls="--", lw=1)
    ax.text(s, -.85, l, ha="center", fontsize=9, color=MUTED)
fig.subplots_adjust(right=.78)
ax.scatter([], [], marker="^", color="white", edgecolor=MUTED, label="Documentation")
ax.scatter([], [], marker="s", color="white", edgecolor=MUTED, label="Quality")
ax.scatter([], [], marker="D", color="white", edgecolor=MUTED, label="Fitness")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.1), ncol=3, fontsize=9)
title(ax, "Two datasets out of eight clear the compliance threshold",
      "Large dot: global score · small symbols: score of each block (out of 100)")
plt.show()

## 5 · Valuation: three approaches, one value

### 5.1 The three amounts per dataset

The three approaches do not tell the same story. The income approach dominates almost everywhere; the market approach remains the most fragile for lack of comparables, hence its lowest confidence.

In [ ]:
r = R.sort_values("after")
fig, ax = plt.subplots(figsize=(11, 5.6))
h, y = .24, np.arange(len(r))
for k, (lab, col) in enumerate(APPROACHES.items()):
    vals = r[["vc", "vm", "vr"][k]] / 1e6
    ax.barh(y + (1 - k) * h, vals, height=h - .03, color=col, label=lab)
ax.scatter(r.after / 1e6, y, marker="|", s=420, color=INK, lw=2, zorder=4, label="Hybrid value retained")
for i, v in enumerate(r.after): ax.text(v/1e6, i + .44, eur(v), ha="center", fontsize=8.5, color=INK)
ax.set_yticks(y, r.index, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel("millions of euros")
ax.legend(loc="lower right", fontsize=9.5)
title(ax, "Income weighs more than cost and market combined",
      "Amount from each approach, and hybrid value retained (black mark), before attribution")
plt.show()

### 5.2 Where the retained value comes from

Contribution of each approach to the hybrid value, once weighted by confidence. This is the **acknowledged weakness** of the model: the income leg weighs most, while it is the one that depends most on assumptions about future benefits.

In [ ]:
r = R.sort_values("part_r")
fig, ax = plt.subplots(figsize=(11, 4.8))
left = np.zeros(len(r))
for k, (lab, col) in enumerate(APPROACHES.items()):
    v = r[["part_c", "part_m", "part_r"][k]].values
    ax.barh(r.index, v, left=left, color=col, height=.62, label=lab, edgecolor=SURFACE, lw=2)
    for i, (g, w) in enumerate(zip(left, v)):
        if w > .07: ax.text(g + w/2, i, f"{w*100:.0f}%", ha="center", va="center", fontsize=9,
                            color="white" if lab != "Income" else INK)
    left += v
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", colors=INK)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(.5, -.09), fontsize=9.5)
title(ax, "The income leg carries most of the retained value",
      "Share of each approach in the hybrid value, after weighting by confidence")
plt.show()

## 6 · Correction: sorting without inflating

Before: a quality factor set by hand, a fixed discount. After: the compliance score drives all three approaches. Result: well-documented datasets gain, the others lose, and the portfolio barely moves.

In [ ]:
r = R.sort_values("gap")
fig, ax = plt.subplots(figsize=(11, 4.8))
for i, (c, row) in enumerate(r.iterrows()):
    col = UP if row.gap >= 0 else DOWN
    ax.plot([row.before/1e6, row.after/1e6], [i, i], color=col, lw=2.2, alpha=.55, solid_capstyle="round")
    ax.scatter(row.before/1e6, i, s=34, color="white", edgecolor=MUTED, lw=1.2, zorder=3)
    ax.scatter(row.after/1e6, i, s=60, color=col, zorder=4)
    ax.text(max(row.before, row.after)/1e6 + .07, i, pct(row.gap), va="center", fontsize=9.5, color=INK2)
ax.set_yticks(range(len(r)), r.index, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel("hybrid value, millions of euros (before attribution)")
ax.scatter([], [], s=34, color="white", edgecolor=MUTED, label="Before (factor set by hand)")
ax.scatter([], [], s=60, color=UP, label="After (driven by the score)")
ax.legend(loc="lower right", fontsize=9.5)
title(ax, f"Each dataset moves by {pct(R.gap.min(),0)} to {pct(R.gap.max(),0)}, the portfolio by {pct(TOT_AF/TOT_BF-1)}",
      "Blue: the dataset gains value · red: it loses value")
plt.show()

## 7 · Attribution: what actually accrues

Four datasets are held in a joint venture. The actual share is not known: the study takes 50% as a prudent assumption and does not try to guess it more finely. Without this correction, joint-venture assets would carry most of the value shown.

In [ ]:
gross = R.after.sum()
losses = (R.after - R.attrib)[R.share < 1].sort_values(ascending=False)
steps = [("Hybrid value\n100%", gross, "total")] + [(f"{c}\n−50%", -v, "delta") for c, v in losses.items()] + [("Attributable\nvalue", TOT_AF, "total")]
fig, ax = plt.subplots(figsize=(11, 4.6))
cum = 0
for i, (lab, v, t) in enumerate(steps):
    if t == "total":
        ax.bar(i, v/1e6, color=BLUE, width=.6); cum = v
        ax.text(i, v/1e6 + .12, eur(v), ha="center", fontsize=10, fontweight="semibold")
    else:
        ax.bar(i, -v/1e6, bottom=(cum + v)/1e6, color=DOWN, alpha=.85, width=.6)
        ax.text(i, cum/1e6 + .12, eur(v), ha="center", fontsize=9, color=INK2)
        cum += v
ax.set_xticks(range(len(steps)), [e[0] for e in steps], fontsize=9.5, color=INK)
ax.grid(axis="x", visible=False); ax.set_ylabel("millions of euros")
jv_share = R.after[R.share < 1].sum() / gross
title(ax, f"The joint-venture share removes {(1 - TOT_AF/gross)*100:.0f}% of the gross value",
      f"Joint-venture assets carry {jv_share*100:.0f}% of the gross value · share retained: 50% (assumption)")
plt.show()

## 8 · Regulatory exposure and priorities

Five criteria scored from 1 to 5 measure the risk that a dataset is one day classified as "important data", and so subject to transfer restrictions. The dominant criterion (30%) is the **maturity of the local industry**: the more it catches up, the more the regulator protects the sector's data.

Priority crosses value (normalised to 100) and exposure, with two thresholds at 50:

| | Exposure < 50 | Exposure ≥ 50 |
|---|---|---|
| **Value ≥ 50** | P2 · act without urgency | **P1 · act first** |
| **Value < 50** | P4 · defer | P3 · monitor and secure |

In [ ]:
w = np.array(list(EXPO_WEIGHTS.values()))
R["expo"] = (EXPO.values @ w) / 5 * 100
R["value_norm"] = R.after / R.after.max() * 100
def prio(v, e):
    if v >= P["thrValue"]: return "P1" if e >= P["thrExpo"] else "P2"
    return "P3" if e >= P["thrExpo"] else "P4"
R["priority"] = [prio(v, e) for v, e in zip(R.value_norm, R.expo)]
LAB = {"P1": "Act first", "P2": "Act without urgency", "P3": "Monitor and secure", "P4": "Defer"}

fig, ax = plt.subplots(figsize=(11, 6.4))
ax.add_patch(plt.Rectangle((50, 50), 55, 55, color=PRIO["P1"], alpha=.07, lw=0))
ax.axvline(P["thrValue"], color=AXIS, ls="--", lw=1); ax.axhline(P["thrExpo"], color=AXIS, ls="--", lw=1)
for q, (x, y) in {"P1": (101, 101), "P2": (101, 4), "P3": (2, 101), "P4": (2, 4)}.items():
    ax.text(x, y, f"{q} · {LAB[q]}", ha="right" if x > 50 else "left", va="top" if y > 50 else "bottom",
            fontsize=9.5, color=PRIO[q], fontweight="semibold")
offset = {"Energy": (8, -12), "Personnel": (8, 6), "Telemetry": (8, -12), "Supply": (8, 8),
          "Suppliers": (8, -12), "Maintenance": (-12, 10), "Process": (12, -16), "Testing": (-8, 10)}
for c, row in R.iterrows():
    size = 60 + row.attrib / R.attrib.max() * 520
    ax.scatter(row.value_norm, row.expo, s=size, color=PRIO[row.priority], alpha=.85, edgecolor="white", lw=2, zorder=3)
    dx, dy = offset[c]
    ax.annotate(c, (row.value_norm, row.expo), xytext=(dx, dy), textcoords="offset points",
                ha="left" if dx > 0 else "right", fontsize=10, color=INK)
ax.set_xlim(0, 104); ax.set_ylim(0, 104)
ax.set_xlabel("normalised value (100 = highest-valued dataset)"); ax.set_ylabel("exposure score (0 to 100)")
title(ax, "Three datasets combine high value and high exposure",
      "Circle size: attributable value · shaded quadrant: priority P1")
plt.show()

recap = R[["verdict","attrib","expo","value_norm","priority"]].copy().sort_values(["priority","attrib"], ascending=[True, False])
recap.insert(0, "Dataset", recap.index); recap["priority"] = recap.priority + " · " + recap.priority.map(LAB)
recap.columns = ["Dataset","Verdict","Attributable value","Exposure","Normalised value","Priority"]
table(recap, {"Attributable value": eur, "Exposure": "{:.0f}", "Normalised value": "{:.0f}"}, bar=["Exposure"])

## 9 · Gains by criterion: where to invest effort

For each dataset, what raising one criterion to 5 would bring, all other assumptions unchanged. The sensitivity of value to the score is computed numerically, then allocated by the weight of each criterion. An empty cell means the criterion is already at its maximum.

In [ ]:
NB = CRITERIA.block.value_counts()
BLOCK_WEIGHT = {"A": P["wDoc"], "B": P["wQual"], "C": P["wApp"]}
G = pd.DataFrame(index=CRITERIA.code, columns=R.index, dtype=float)
for _, j in DATASETS.iterrows():
    s = R.score[j.short]
    sens = value(j, s + 1)["val"] - value(j, s)["val"]          # € per score point
    for _, c in CRITERIA.iterrows():
        G.loc[c.code, j.short] = (5 - SCORES.loc[c.code, j.short]) * BLOCK_WEIGHT[c.block] * 25 / NB[c.block] * sens
G = G[R.score.sort_values().index]
fig, ax = plt.subplots(figsize=(11.5, 7.2))
Gm = np.ma.masked_where(G.values <= 0, G.values / 1e3)
ax.imshow(Gm, cmap=mpl.colors.LinearSegmentedColormap.from_list("b", ["#e9f1fb", "#6da7ec", "#0d366b"]), aspect="auto")
for (i, k), v in np.ndenumerate(G.values):
    if v > 0: ax.text(k, i, f"{v/1e3:.0f}", ha="center", va="center", fontsize=9, color="white" if v/1e3 > G.values.max()/1e3*.45 else INK)
ax.set_xticks(range(G.shape[1]), G.columns, color=INK); ax.xaxis.tick_top()
ax.set_yticks(range(17), [f"{c}  {n}" for c, n in zip(CRITERIA.code, CRITERIA["criterion"])], color=INK2, fontsize=9.5)
ax.grid(False); ax.tick_params(length=0)
for sp in ax.spines.values(): sp.set_visible(False)
for y in (3.5, 11.5): ax.axhline(y, color=SURFACE, lw=4)
fig_title(fig, ax, "What raising each criterion to 5 would bring, in thousands of euros",
          "Columns from least to most compliant dataset · value of the dataset, before attribution · empty cell: already at 5")
plt.show()

top = G.stack().sort_values(ascending=False).head(8).reset_index()
top.columns = ["Criterion", "Dataset", "Gain"]
top["Criterion"] = top["Criterion"] + " · " + top["Criterion"].map(dict(zip(CRITERIA.code, CRITERIA["criterion"])))
display(Markdown("**The eight most profitable improvement actions in the portfolio**"))
table(top[["Dataset", "Criterion", "Gain"]], {"Gain": eur}, bar=["Gain"])

## 10 · Sensitivity: which assumptions matter

Each bar shows the attributable value of the portfolio when a single assumption moves to its low then high bound, all others held fixed. The longest bars mark the assumptions to defend first in front of a sceptical audience.

In [ ]:
def portfolio(Pm=None, mult=None):
    Px = {**P, **(Pm or {})}; D = DATASETS.copy()
    for col, m in (mult or {}).items(): D[col] = D[col] * m
    return compute(Px, D).attrib.sum()

base = portfolio()
tests = [
    ("Discount rate (10% / 15%)", {"rate": .15}, None, {"rate": .10}, None),
    ("Profit and savings (−30% / +30%)", None, {"profit": .7, "savings": .7}, None, {"profit": 1.3, "savings": 1.3}),
    ("Confidence in income (0.55 / 0.85)", {"cr": .55}, None, {"cr": .85}, None),
    ("Income weight (35% / 55%)", {"pr": .35, "pc": .35, "pm": .30}, None, {"pr": .55, "pc": .25, "pm": .20}, None),
    ("Reference price (−30% / +30%)", None, {"price": .7}, None, {"price": 1.3}),
    ("Rebuild cost (−20% / +20%)", None, {"fte": .8, "infra": .8}, None, {"fte": 1.2, "infra": 1.2}),
    ("Score effect on income β (0.25 / 0.75)", {"beta": .25}, None, {"beta": .75}, None),
    ("Score effect on discount α (0.15 / 0.45)", {"alpha": .15}, None, {"alpha": .45}, None),
]
rows = []
for lab, pb, mb, ph, mh in tests:
    rows.append((lab, portfolio(pb, mb) - base, portfolio(ph, mh) - base))
S = pd.DataFrame(rows, columns=["assumption", "low", "high"])
S["amp"] = (S.high - S.low).abs(); S = S.sort_values("amp")
fig, ax = plt.subplots(figsize=(11, 4.9))
for i, row in enumerate(S.itertuples()):
    for v in (row.low, row.high):
        ax.barh(i, v/1e6, color=UP if v >= 0 else DOWN, height=.58)
        ax.text(v/1e6 + (.03 if v >= 0 else -.03), i, pct(v/base, 0), va="center",
                ha="left" if v >= 0 else "right", fontsize=9, color=INK2)
ax.axvline(0, color=INK, lw=1)
m = max(S.low.abs().max(), S.high.abs().max()) / 1e6; ax.set_xlim(-m * 1.22, m * 1.22)
ax.set_yticks(range(len(S)), S.assumption, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel(f"gap to the central value ({eur(base)}), millions of euros")
title(ax, "Income assumptions dominate the result",
      "One assumption at a time · blue: value rises · red: value falls")
plt.show()

## 11 · Monte Carlo: a range rather than a point

10,000 draws. Each uncertain assumption follows a triangular distribution around its central value; the compliance score is held fixed to isolate economic uncertainty. The seed is fixed: the result is reproducible.

| Assumption | Low | Central | High |
|---|---|---|---|
| FTE-months, infrastructure | ×0.80 | ×1 | ×1.25 |
| Cost of an FTE | ×0.90 | ×1 | ×1.10 |
| Volume | ×0.80 | ×1 | ×1.20 |
| Reference price | ×0.60 | ×1 | ×1.50 |
| Discount | −10 pts | 0 | +10 pts |
| Profit, savings | ×0.60 | ×1 | ×1.30 |
| Discount rate | 10% | 12% | 15% |

In [ ]:
rng = np.random.default_rng(2026)
N = 10_000
tri = lambda a, m, b, n=N: rng.triangular(a, m, b, n)
sims = {}
for _, j in DATASETS.iterrows():
    s = R.score[j.short]
    f    = clip(P["fMin"] + (s - 20) / 80 * (P["fMax"] - P["fMin"]), P["fMin"], P["fMax"])
    disc = clip(j.disc - P["alpha"]*(s - 50)/100 + tri(-.10, 0, .10), P["discMin"], P["discMax"])
    mul  = 1 + P["beta"]*(s - 50)/100
    t    = tri(.10, .12, .15)
    vc = (j.fte*tri(.8,1,1.25)*j.fte_cost*tri(.9,1,1.1) + j.infra*tri(.8,1,1.25)) * f
    vm = j.vol*tri(.8,1,1.2) * j.price*tri(.6,1,1.5) * (1 - disc)
    vr = (j.profit*tri(.6,1,1.3) + j.savings*tri(.6,1,1.3)) * mul * (1 - (1+t)**(-j.horizon)) / t
    wc, wm, wr = P["pc"]*P["cc"], P["pm"]*j.mkt_conf, P["pr"]*P["cr"]
    sims[j.short] = (wc*vc + wm*vm + wr*vr) / (wc + wm + wr) * j.share
SIM = pd.DataFrame(sims); PF = SIM.sum(axis=1)
q = SIM.quantile([.05, .5, .95]).T.sort_values(.5)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
for i, (c, row) in enumerate(q.iterrows()):
    a1.plot([row[.05]/1e6, row[.95]/1e6], [i, i], color="#9ec5f4", lw=7, solid_capstyle="round")
    a1.scatter(row[.5]/1e6, i, s=55, color=BLUE, edgecolor="white", lw=1.5, zorder=3)
    a1.text(row[.95]/1e6 + .04, i, eur(row[.5]), va="center", fontsize=9, color=INK2)
a1.set_yticks(range(len(q)), q.index, color=INK); a1.grid(axis="y", visible=False)
a1.set_xlabel("attributable value, millions of euros")
title(a1, "90% range per dataset", "Bar: 5th to 95th percentile · dot: median")

a2.hist(PF/1e6, bins=60, color="#6da7ec", edgecolor=SURFACE, lw=.4)
for v, l in ((PF.quantile(.05), "P5"), (PF.median(), "median"), (PF.quantile(.95), "P95")):
    a2.axvline(v/1e6, color=INK if l == "median" else MUTED, lw=1.2, ls="-" if l == "median" else "--")
    a2.text(v/1e6, a2.get_ylim()[1]*.97, f"{l}\n{eur(v)}", ha="center", va="top", fontsize=9, color=INK,
            bbox=dict(fc=SURFACE, ec="none", pad=1.5))
a2.set_xlabel("attributable value of the portfolio, millions of euros"); a2.set_yticks([])
a2.spines["left"].set_visible(False); a2.grid(axis="y", visible=False)
title(a2, "The portfolio over 10,000 scenarios", f"Central model value: {eur(TOT_AF)}")
plt.tight_layout(); plt.show()

print(f"Portfolio: median {eur(PF.median())} · 90% interval [{eur(PF.quantile(.05))} ; {eur(PF.quantile(.95))}]"
      f" · relative width {((PF.quantile(.95)-PF.quantile(.05))/PF.median())*100:.0f}%")

**Reading.** The simulated median sits slightly below the central model value: benefit assumptions are deliberately asymmetric (down to −40%, but only +30% up), because a benefit plan disappoints more often than it surprises. The ranges of neighbouring datasets overlap strongly: the order of the top two or three is robust, that of the middle datasets much less so. One more reason to present a **ranking by groups** (P1 to P4) rather than an exact rank.

## 12 · Weighting: deriving the weights instead of setting them

The 30 / 25 / 45 weights are set by judgement. The DATAMITE framework (Cork, Aachen) derives them by **pairwise comparisons** (ANP, of which AHP is the case without interdependence). The principle: the approaches are compared two by two on Saaty's scale (1 = equal importance, 3 = moderately more, 5 = strongly more), the principal eigenvector of the matrix gives the weights, and a **consistency ratio** below 10% ensures the judgements do not contradict each other.

Two principles guide the application:
- the level of aggregation depends on the use: **per dataset** if the data is sold, **at portfolio level** if it serves an internal service. Here the use is internal, so one matrix for the whole portfolio;
- beyond about ten compared elements, comparisons can be "smoothed" to absorb inconsistencies.

The matrix below is **illustrative**: it translates the implicit judgements of the study and should be filled in with domain experts.

In [ ]:
names = ["Cost", "Market", "Income"]
A = np.array([[1,   2,   1/2],     # cost is more reliable than market, less relevant than income
              [1/2, 1,   1/3],
              [2,   3,   1  ]])
ev, vec = np.linalg.eig(A)
k = np.argmax(ev.real); w_ahp = np.abs(vec[:, k].real); w_ahp /= w_ahp.sum()
lam = ev.real[k]; n = len(A); CI = (lam - n) / (n - 1); RI = {3: .58, 4: .90, 5: 1.12}[n]; CR = CI / RI

display(Markdown(f"**Consistency ratio: {CR*100:.1f}%** ({'consistent' if CR < .1 else 'to review'}, threshold 10%) · λ max = {lam:.3f}"))
w_jud = np.array([P["pc"], P["pm"], P["pr"]])
v_ahp = compute({**P, "pc": w_ahp[0], "pm": w_ahp[1], "pr": w_ahp[2]})

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.25]})
x = np.arange(3)
a1.bar(x - .19, w_jud, width=.36, color=MUTED, label="Judgement (study)")
a1.bar(x + .19, w_ahp, width=.36, color=[BLUE, ORANGE, AQUA], label="Derived (AHP)")
for i in range(3):
    a1.text(i - .19, w_jud[i] + .01, f"{w_jud[i]*100:.0f}%", ha="center", fontsize=9, color=INK2)
    a1.text(i + .19, w_ahp[i] + .01, f"{w_ahp[i]*100:.0f}%", ha="center", fontsize=9, color=INK)
a1.set_xticks(x, names, color=INK); a1.set_yticks([]); a1.spines["left"].set_visible(False); a1.grid(False)
a1.legend(fontsize=9, loc="upper left")
title(a1, "Weights set and weights derived", "Grey: judgement · colour: pairwise comparisons")

comp = pd.DataFrame({"Judgement": R.attrib, "AHP": v_ahp.attrib}).sort_values("Judgement")
for i, (c, row) in enumerate(comp.iterrows()):
    a2.plot([row.Judgement/1e6, row.AHP/1e6], [i, i], color=GRID, lw=2)
    a2.scatter(row.Judgement/1e6, i, s=36, color=MUTED, zorder=3)
    a2.scatter(row.AHP/1e6, i, s=48, color=BLUE, zorder=4)
a2.set_yticks(range(len(comp)), comp.index, color=INK); a2.grid(axis="y", visible=False)
a2.set_xlabel("attributable value, millions of euros")
rank_j, rank_a = R.attrib.rank(ascending=False), v_ahp.attrib.rank(ascending=False)
moved = [c for c in R.index if rank_j[c] != rank_a[c]]
prio_a = [prio(v, e) for v, e in zip(v_ahp.after / v_ahp.after.max() * 100, R.expo)]
same_prio = list(R.priority) == prio_a
t2 = "The ranking does not move" if not moved else f"{len(moved)} datasets swap ranks, priorities {'hold' if same_prio else 'change'}"
title(a2, t2, f"Portfolio: {eur(R.attrib.sum())} → {eur(v_ahp.attrib.sum())}")
plt.tight_layout(); plt.show()

max_gap = ((v_ahp.attrib / R.attrib) - 1).abs().max()
print("Datasets changing rank:", ", ".join(f"{c} ({rank_j[c]:.0f} → {rank_a[c]:.0f})" for c in moved) or "none")
print("P1 to P4 groups identical:", "yes" if same_prio else "no", "· largest gap per dataset:", pct(max_gap))

**Reading.** With consistent judgements, pairwise comparisons give more weight to income than the study does (54% against 45%) and less to market. Total value therefore rises mechanically, a reminder that **an amount depends on the weights, not only on the data**. The result that matters lies elsewhere: the priority groups do not move, and only two datasets of very close value swap ranks. The decision holds whatever the weighting method.

## 13 · External anchors

### 13.1 Against the national average ticket

Public statistics as of June 2025: 3,364 datasets listed on the data exchanges for about 4 billion yuan of cumulative transactions, an **average ticket of about €154k** per dataset (at 7.73 yuan to the euro). This comparable pools public and private data of every kind: it gives an order of magnitude, not a price.

In [ ]:
TICKET = 4e9 / 3364 / 7.73
mult = (R.attrib / TICKET).sort_values()
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.barh(mult.index, mult.values, color=[BLUE if m < 10 else "#0d366b" for m in mult], height=.6)
for i, m in enumerate(mult): ax.text(m + .15, i, f"×{m:.1f}", va="center", fontsize=9.5, color=INK2)
ax.axvline(1, color=INK, lw=1); ax.text(1.1, len(mult) - .45, "national average ticket", fontsize=9, color=INK2)
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", colors=INK)
ax.set_xlabel("multiple of the national average ticket (attributable value / €154k)")
title(ax, f"The portfolio is worth on average ×{(R.attrib.mean()/TICKET):.1f} the national ticket",
      "A gap to state, not to hide: specialised industrial datasets against an all-category average")
plt.show()

### 13.2 The gap between perceived and recognised value

The survey by Gao, Liu and Lu (version of 13 September 2026) of Chinese listed companies measures what companies *think* their data is worth, and what they *actually* book on the balance sheet since the rule allows it (January 2024).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
lab = ["Estimated value\nat historical cost", "Estimated value\nat economic value", "Amount actually\nbooked"]
val = [12.15, 17.15, 0.26]
ax.bar(lab, val, color=["#6da7ec", BLUE, CRIT], width=.5)
for i, v in enumerate(val): ax.text(i, v + .4, f"{v:.2f}%", ha="center", fontsize=12, fontweight="semibold")
ax.set_ylabel("share of total assets"); ax.set_ylim(0, 20); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", colors=INK)
title(ax, "The value of data lives almost entirely off the balance sheet",
      "Chinese listed companies · 132 of over 5,000 (2.6%) had booked data by end-2025 · Gao, Liu and Lu (2026)")
plt.show()

**What this changes for the study.** The accounting route is narrow: the rule is prospective (spending before 2024 is lost for the balance sheet), and recognition runs into valuation and rights. The grid therefore sits on the ground of **managerial valuation**: rank, prioritise, decide what to document first. That is where value can be mobilised today.

One complementary benchmark:
- as of 30 April 2026, **136 A-share companies** had booked **3.79 billion yuan** of data, 82% as intangible assets; the three telecom carriers hold 55% of it, and manufacturing has the largest number of companies (32).

### 13.3 Market context

Two recent signals, from outside the jurisdiction studied, point the same way as the grid: the value of data is judged first on its quality, governance and readiness for use.

- **An industry grid of the same shape.** In May 2026 Black Book Research published a European data valorisation index for healthcare IT, built from 700 hospital IT leaders in eight markets on 18 indicators: data capture, interoperability, quality, governance, analytics, research and AI readiness, localisation, implementation discipline. As here, value is scored criterion by criterion before it is priced.
- **A regulatory clock that pushes for readiness.** The European Health Data Space sets March 2027 for the adoption of several implementing acts, before application phases in 2029 and 2031 (timeline as reported by Black Book).
- **A market taking shape.** MarketsandMarkets puts the healthcare data valorisation market at USD 0.7 billion in 2026 and USD 1.3 billion in 2031, 14.5% a year; clinical data is the largest share, cloud the fastest-growing segment.

> **Read with care.** These are research-vendor sources, on another sector than the grid's, and not audited. They give an order of magnitude and a check on method, not a price comparable.

## 14 · Extension: the value of future flows from an installed base

The current portfolio says nothing of the data that the equipment delivered tomorrow will produce. An order of magnitude, with **purely illustrative** assumptions, deliberately simple and all adjustable: a capturable value per unit and per year, a lifetime, a realistic capture rate.

In [ ]:
H = dict(value_unit_yr=20_000, life=15, rate=.12, capture=.35, order=10, fleet=1_000)   # illustrative parameters
fa = annuity(H["life"], H["rate"])
per_unit = H["value_unit_yr"] * fa * H["capture"]
cards = [("per unit delivered", eur(per_unit), "discounted capturable value over the lifetime"),
         (f"for an order of {H['order']} units", eur(per_unit * H["order"]), "the data layer hidden behind the sale"),
         ("annual flow from the installed base", eur(H["value_unit_yr"] * H["capture"] * H["fleet"]), f"{H['fleet']:,} units in service, at the capture rate")]
fig = plt.figure(figsize=(12, 2.6))
for i, (l, v, s) in enumerate(cards):
    ax = fig.add_axes([.005 + i*.333, .05, .32, .88]); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=.06", transform=ax.transAxes, fc=CARD, ec="none"))
    ax.text(.07, .64, v, fontsize=24, fontweight="semibold", color=BLUE, transform=ax.transAxes)
    ax.text(.07, .36, l, fontsize=11.5, transform=ax.transAxes)
    ax.text(.07, .16, s, fontsize=9.5, color=MUTED, transform=ax.transAxes)
plt.show()

cap = np.linspace(.1, .6, 51)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(cap*100, H["value_unit_yr"] * fa * cap * H["order"] / 1e6, color=BLUE, lw=2)
ax.axvline(H["capture"]*100, color=MUTED, ls="--", lw=1)
ax.text(H["capture"]*100 + .6, ax.get_ylim()[1]*.9 if ax.get_ylim()[1] else 1, f"assumption retained: {H['capture']*100:.0f}%", fontsize=9, color=INK2)
ax.set_xlabel("capture rate (%)"); ax.set_ylabel("millions of euros")
title(ax, "The data layer of an order depends first on the capture rate",
      f"Order of {H['order']} units · {eur(H['value_unit_yr'])} per unit per year · {H['life']} years at {H['rate']*100:.0f}%")
plt.show()

> This future value stays **outside the grid**: it depends on commercial assumptions, not on the quality of existing datasets. It shows the order of magnitude of what is at stake, not a figure for a decision.

## 15 · Regulatory watch: the timeline up to October 2026

The sequence matters more than any single text: the State **took stock before setting prices**, and each step followed the previous one within months. All public pricing schemes cover **public data**; corporate data is left to free negotiation.

In [ ]:
EV = [
 ("2020-04", "Data becomes a factor of production", "Doctrine", MUTED),
 ("2022-12", '"Twenty measures": separate rights over data', "Doctrine", MUTED),
 ("2023-08", "Accounting rule: data can enter the balance sheet", "Accounting", BLUE),
 ("2023-10", "National Data Administration created", "Institutions", MUTED),
 ("2024-05", "First negative list for data exports (free trade zone)", "Transfers", ORANGE),
 ("2024-08", "Municipal negative list, strategic sectors included", "Transfers", ORANGE),
 ("2024-10", 'Framework text "1" on public data', "Public data", AQUA),
 ("2025-03", 'The three "3" texts enter into force (registry, licence, price)', "Public data", AQUA),
 ("2025-08", "Guidelines: high-quality datasets (17 criteria)", "Quality", BLUE),
 ("2026-03", '"Industrial data foundations" action, horizon end-2026', "Industry", BLUE),
 ("2026-06", "New data-backed securitisations suspended", "Finance", CRIT),
 ("2026-07", "Guide to registering property rights (pilot)", "Rights", BLUE),
 ("2026-08", "National registry: over 300,000 entries · US Congress report", "Public data", AQUA),
 ("2026-09", "National standards for classifying and registering data assets (1 Sept.)", "Accounting", BLUE),
 ("2026-09", "National rights registration system online: 56 certificates on day one", "Rights", BLUE),
 ("2026-10", 'National final of the "Data Element ×" competition (20 to 23 Oct.)', "Ecosystem", MUTED),
]
ev = pd.DataFrame(EV, columns=["date", "event", "theme", "c"])
ev["d"] = pd.to_datetime(ev.date)
fig, ax = plt.subplots(figsize=(12, 8.4)); ax.axis("off")
n = len(ev); ys = np.arange(n)[::-1]
ax.plot([0.12, 0.12], [-.5, n - .5], color=AXIS, lw=1.2)
prev_year = None
for y, row in zip(ys, ev.itertuples()):
    ax.scatter(.12, y, s=70, color=row.c, edgecolor="white", lw=1.6, zorder=3)
    yr = row.date[:4]
    ax.text(.0, y, row.d.strftime("%m.%Y"), va="center", fontsize=9.5, color=INK if yr != prev_year else MUTED)
    prev_year = yr
    ax.text(.15, y, row.event, va="center", fontsize=10.5, color=INK)
    ax.text(.98, y, row.theme, va="center", ha="right", fontsize=9, color=MUTED)
ax.set_xlim(0, 1); ax.set_ylim(-.8, n)
ax.text(0, n + .1, "From principle to infrastructure: six years of construction", fontsize=14, fontweight="semibold")
plt.show()

**The two developments of September and October 2026**

1. **A national system for registering property rights** has been online since 11 September, in a pilot phase: 56 certificates issued on day one, valid for five years, filed through three national data exchanges and valid across the country. It is the missing link between quality (the grid) and assetisation.
2. **On international accounting standards**, the IASB project on intangible assets is due to set its direction in the second half of 2026. Data is not named at this stage.

> **Scope caveat.** Neither the public registry, nor licensed operation, nor administered pricing apply directly to a private company's data. They provide a **benchmark layer**, not a valuation method.

## 16 · Convergence with the state of the art

Three recent works describe the architecture the grid rebuilt for an anonymised case.

In [ ]:
L = pd.DataFrame([
 ("Method families", "Cost, market, income", "Cost, market, income (408 papers)", "Costs, demand, price in Val", "Cost imposed on balance-sheet entry"),
 ("Structure", "Compliance → value → exposure", "Call for hybrid frameworks", "Screening → metrics → calculation", "Survey + archival data"),
 ("Core dimensions", "Documentation, quality, fitness", "Quality as a critical component", "Quality, Relevance, Utility", "Valuation and rights as obstacles"),
 ("Weighting", "Judgement (30/25/45)", "Not specified", "ANP (pairwise comparisons)", "Not applicable"),
 ("Nature of value", "Relative, for ranking", "No standard method", "Value, not a price", "Perceived ≫ recognised"),
 ("Empirical validation", "Anonymised case, 8 datasets", "Missing (finding)", "Missing (finding)", "132 listed companies"),
], columns=["Component", "The grid", "Rennes · Mohan et al. 2026", "Cork · Vyhmeister et al. Dec. 2025", "Gao, Liu and Lu 2026"])
table(L)

**What the convergence proves, and what it does not.** Two independent teams describe the same architecture as the grid: an argument for **methodological legitimacy**. It is not proof that the amounts are right. The contribution of this work lies precisely there: applying the framework to an anonymised case, which both reviews identify as the main gap in the field.

## 17 · Limits and next steps

| Limit | What it implies | Planned response |
|---|---|---|
| Relative values, not prices | Never say "this dataset is worth X" | External benchmark for any price range |
| Income leg dominates | The result depends on benefit assumptions | Sensitivity (part 10) and Monte Carlo (part 11) |
| Weights set by judgement | The "why 45%?" objection | AHP matrix filled in by domain experts |
| Joint-venture share unknown | 50% retained for prudence | Replace with the actual share |
| Exposure scored by expert judgement | Indicative score, not regulatory | Check against the sector's important-data catalogues |
| Productisation step not covered | The grid jumps from inventory to value | A productisation layer (valuation canvas) to build |
| Assetisation not measured | A high score ≠ an asset | Rights registration and accounting treatment to study |
| Local standards ≠ IFRS | A booking under local standards does not carry over as is into consolidation | Check the IFRS consolidation treatment |

## Export

The next cell saves the results to `outputs/valuation_results.csv`, for use in a spreadsheet.

In [ ]:
out = R[["dataset","entity","share","score","verdict","vc","vm","vr","before","after","gap","attrib","expo","priority"]].copy()
out.columns = ["dataset","ownership","share","score","verdict","value_cost","value_market","value_income",
               "hybrid_before","hybrid_after","gap","attributable","exposure","priority"]
import os
os.makedirs("../outputs", exist_ok=True)
out.round(4).to_csv("../outputs/valuation_results.csv")
print("Written: outputs/valuation_results.csv")

## Sources

**Academic**
- Mohan, S. K., Bharathy, G. and Jalan, A. (2026). *Enterprise Data Valuation: A Targeted Literature Review*. Journal of Economic Surveys, 40(1), 73-92. DOI 10.1111/joes.12705.
- Vyhmeister, E., Pietropaoli, B., Bub, U., Schneider, R. and Visentin, A. (Dec. 2025). Data valuation framework of the Horizon Europe project DATAMITE (101092989). University College Cork, FIR e.V. RWTH Aachen. arXiv:2512.07664. https://arxiv.org/abs/2512.07664
- Gao, X., Liu, J. and Lu, H. (2026). *Accounting for Data Assets*. SSRN 5164215, version of 13 September 2026.
- Saaty, T. L. (2006). *The Analytic Network Process*. Springer.

**Institutional and press**
- U.S.-China Economic and Security Review Commission (18 August 2026). *The People's Republic of Data: How China Is Turning Data into Capital*. https://www.uscc.gov/research/peoples-republic-data-how-china-turning-data-capital
- National Data Administration, press conference of 25 August 2026 (national registry, 300,000+ entries).
- Xinhua (12 September 2026). Launch of the national system for registering data property rights. https://www.news.cn/fortune/20260912/63c960fce4cd4d6bbeb41f0d542d6370/c.html
- Securities Times (24 June 2026). 136 A-share companies, 3.786 billion yuan of data booked. https://stcn.com/article/detail/3976954.html
- National Data Administration (9 October 2026). Final of the "Data Element ×" competition. https://www.nda.gov.cn/sjj/zwgk/tzgg/1009/20261009165059083009494_pc.html
- IFRS Foundation (April 2026). Update on the *Intangible Assets* project. https://www.ifrs.org/content/dam/ifrs/meetings/2026/april/ac/ac4-iasb-update.pdf
- China Briefing (2025). *The "Negative Lists" Easing Data Export in China's FTZs*. https://www.china-briefing.com/news/china-data-export-policy-ftzs/
- Black Book Research (18 May 2026). *Europe's Healthcare IT Market Enters the 2027 Readiness Cycle*, ACCESS Newswire, Copenhagen (European data valorisation index, 700 respondents, 18 indicators).
- MarketsandMarkets (30 September 2026). Report on the healthcare data market, PR Newswire release 302894117 (USD 0.7 billion in 2026, USD 1.3 billion in 2031, 14.5% a year).

## Reproducibility & License

- **Running:** *Run All* is enough; no external data is read. Python 3.13, `numpy`, `pandas`, `matplotlib` (versions in `requirements.txt`).
- **Seed:** the Monte Carlo simulation (part 11) uses `np.random.default_rng(2026)`; results are identical from one run to the next.
- **Nature of the figures:** working assumptions, consistent with each other, on an anonymised case; relative ranking, not an opposable price.
- **License:** code under the MIT License; text, figures and data under CC BY 4.0.
- **Links:** [GitHub repository](https://github.com/Kukai-7/industrial-data-valuation) · [synthetic dataset on Hugging Face](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · [demonstration](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)